###Env to use: birdnet

In [1]:
import librosa
from skimage import feature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf
import os
from pathlib import Path
from sklearn.metrics import average_precision_score


In [11]:
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
from datetime import datetime

# Load and initialize the BirdNET-Analyzer models.
analyzer = Analyzer()

c:\Users\dgnhk\anaconda3\envs\birdnet\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)



Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.


Predict for positive segments first

In [12]:
audio_folder='../data/Holderried/selections_wavs/'
data_folder='../data/Holderried/'
df = pd.read_csv(data_folder+"selections.csv")

In [4]:
df.annotation.value_counts()

whistle                      2018
croak                         271
woodcock                      193
cwhistle, chase                18
cwoodcock, whistle             18
cwoodcock, chase                8
chase                           6
cwoodcock, whistle, croak       6
ccroak, whistle                 5
cchase, woodcock                2
Name: annotation, dtype: int64

In [5]:
df[df.annotation=="cwhistle, chase"]

,selec,deploy.id,sound.files,view,channel,start,end,bottom.freq,top.freq,species.code,common.name,annotation,recording.id
12,23,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,205.804715,208.804715,3.2821,10.2154,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
13,25,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,207.910150,210.910150,3.0769,10.1333,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
15,24,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,207.006225,210.006225,3.3231,9.9692,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
16,26,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,210.318915,213.318915,3.9385,8.4103,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
18,28,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,212.594680,215.594680,3.4051,8.9026,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
19,27,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,210.796065,213.796065,3.5282,10.4205,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
20,22,14569_1_2020_06_11_FVA133,20200611_193504.WAV,Spectrogram 1,1,204.902150,207.902150,3.2000,10.4615,eurwoo,Eurasian Woodcock,"cwhistle, chase",11601
378,411,14569_1_2020_06_11_FVA133,20200615_203516.WAV,Spectrogram 1,1,238.259860,241.259860,2.7897,7.5077,eurwoo,Eurasian Woodcock,"cwhistle, chase",11643
379,410,14569_1_2020_06_11_FVA133,20200615_203516.WAV,Spectrogram 1,1,237.216450,240.216450,3.0359,7.3846,eurwoo,Eurasian Woodcock,"cwhistle, chase",11643
380,409,14569_1_2020_06_11_FVA133,20200615_203516.WAV,Spectrogram 1,1,236.376820,239.376820,2.9949,7.1795,eurwoo,Eurasian Woodcock,"cwhistle, chase",11643


In [6]:
df_sub = df[df.annotation=="cwhistle, chase"]

In [7]:
file_ids = df_sub.selec.values.tolist()
wav_files = [str(file)+'.wav' for file in file_ids]

In [9]:
predictions = dict.fromkeys(wav_files, 0.0)

In [13]:
for file in wav_files:
    audio_path = Path(os.path.join(audio_folder, file))
    recording = Recording(
        analyzer,
        audio_path,
        min_conf=0.0
    )
    recording.analyze()
    for detection in recording.detections:
        if detection['scientific_name']=='Scolopax rusticola':
            predictions[file] = detection['confidence']

read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 23.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 25.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 24.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 26.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 28.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 27.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 22.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 411.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 410.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 409.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recording 408.wav
read_audio_data
read_audio_data: complete, read  1 chunks.
analyze_recor

In [14]:
predictions

{'23.wav': 0.6704295873641968,
 '25.wav': 0.6080445647239685,
 '24.wav': 0.7185490727424622,
 '26.wav': 0.9951303005218506,
 '28.wav': 0.48598599433898926,
 '27.wav': 0.9923005104064941,
 '22.wav': 0.16404537856578827,
 '411.wav': 0.0,
 '410.wav': 0.023127390071749687,
 '409.wav': 0.0,
 '408.wav': 0.0,
 '407.wav': 0.021061740815639496,
 '406.wav': 0.023272927850484848,
 '405.wav': 0.02072623372077942,
 '404.wav': 0.30997735261917114,
 '403.wav': 0.09104003012180328,
 '402.wav': 0.14040638506412506,
 '401.wav': 0.06744281202554703}

In [15]:
np.mean(list(predictions.values()))

0.29619668227516943